# Daily Gold Price Forecasting — ARIMA / SARIMA / GARCH
**Course:** Applied Time Series Analysis (702DB0E007) · B.Tech AI, Sem VII · MPSTME
**Team members:** Priyanshu Modak (I080), Vishesh Dokania (I019), Parth Zanwar (I074)

This notebook follows the 11-step *Model Deployment* pipeline from the course slides:

| # | Step | Section |
|---|------|---------|
| 1 | Data collection | §1 |
| 2 | Exploratory data analysis | §2 |
| 3 | Missing values | §3 |
| 4 | Stationarity, ADF test | §4 |
| 5 | Transformations | §5 |
| 6 | Seasonality & decomposition | §6 |
| 7 | ACF / PACF | §7 |
| 8 | Identify AR / MA / ARMA / ARIMA / SARIMA | §8 |
| 9 | Identify lags | §9 |
| 10 | AIC / BIC | §9 |
| 11 | Deploy the model | §12 (+ `app.py`) |

Extra sections: model evaluation vs. a naive baseline (§10), residual diagnostics (§11), GARCH for volatility clustering (§11b).

> **How to run:** `pip install -r requirements.txt` then *Run All*. The notebook downloads live gold futures data (`GC=F`) from Yahoo Finance.
> If you are offline it falls back to `data/gold_daily.csv`, otherwise it stops and asks for a real CSV; simulated prices are never used automatically.

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model

# Keep fitting and convergence warnings visible.
plt.rcParams.update({
    "figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})
for d in ["data", "figures", "models", "outputs"]:
    Path(d).mkdir(exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")

# ---- configuration (edit here) ----
TICKER      = "GC=F"        # COMEX gold futures. Alternative: "GLD" (gold ETF)
START       = "2015-01-01"
TEST_DAYS   = 100           # last ~5 months of trading days held out for evaluation
HORIZON     = 30            # business days to forecast ahead
MAX_P, MAX_Q = 4, 4         # ARIMA grid-search bounds
SEAS_PERIOD = 5             # trading-week seasonality (5 business days)
SEED        = 42
np.random.seed(SEED)

## 1. Data Collection

In [ ]:
DATA_PATH = Path("data/gold_daily.csv")

def load_gold():
    # 1) live download
    try:
        import yfinance as yf
        yf.set_tz_cache_location(str(Path("data/.yf_cache").resolve()))
        df = yf.download(TICKER, start=START, auto_adjust=False, progress=False, timeout=20)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])
        if len(df) > 500:
            df.to_csv(DATA_PATH)
            return df, f"Yahoo Finance live ({TICKER})"
        print("yfinance returned too little data.")
    except Exception as e:
        print("yfinance download failed:", repr(e)[:150])
    # 2) cached csv
    if DATA_PATH.exists():
        df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
        return df, f"cached file {DATA_PATH}"
    # Never silently replace real observations with simulated prices.
    raise RuntimeError("No real gold data available. Connect to the internet or supply data/gold_daily.csv.")

raw, DATA_SOURCE = load_gold()
raw.index = pd.to_datetime(raw.index)
raw = raw.sort_index()
raw = raw.loc[~raw.index.duplicated(keep="last")]
raw["Close"] = pd.to_numeric(raw["Close"], errors="coerce")
raw = raw.replace([np.inf, -np.inf], np.nan)
raw.loc[raw["Close"] <= 0, "Close"] = np.nan
print(f"Data source : {DATA_SOURCE}")
print(f"Rows        : {len(raw):,}   ({raw.index.min().date()} → {raw.index.max().date()})")
if DATA_SOURCE == "SYNTHETIC":
    print("\n" + "!" * 70 + "\n!!  SYNTHETIC DEMO DATA — NOT REAL GOLD PRICES. Get internet access\n"
          "!!  (or place a real data/gold_daily.csv) before using any result.\n" + "!" * 70)
raw.tail()

## 2. Exploratory Data Analysis

In [ ]:
print(raw.info())
raw[["Open", "High", "Low", "Close"]].describe().round(2)

In [ ]:
close_raw = raw["Close"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(close_raw, lw=1, label="Close")
ax[0].plot(close_raw.rolling(252).mean(), lw=1.5, color="crimson", label="252-day mean")
ax[0].set_title("Gold futures — daily closing price (USD / oz)"); ax[0].legend()
ax[1].plot(np.log(close_raw), lw=1, color="darkgoldenrod")
ax[1].set_title("Log price")
savefig("01_line_plot"); plt.show()

In [ ]:
r_raw = np.log(close_raw).diff().dropna()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(close_raw.shift(1), close_raw, s=4, alpha=.4)
ax[0].set_xlabel("Price (t-1)"); ax[0].set_ylabel("Price (t)"); ax[0].set_title("Price vs previous-day price")
ax[1].scatter(r_raw.shift(1), r_raw, s=4, alpha=.4, color="darkorange")
ax[1].set_xlabel("Return (t-1)"); ax[1].set_ylabel("Return (t)"); ax[1].set_title("Return vs previous-day return")
savefig("02_scatter"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].hist(close_raw, bins=40, color="goldenrod", edgecolor="white"); ax[0].set_title("Histogram — price")
ax[1].hist(r_raw, bins=60, density=True, color="steelblue", edgecolor="white", alpha=.8)
xs = np.linspace(r_raw.min(), r_raw.max(), 300)
ax[1].plot(xs, stats.norm.pdf(xs, r_raw.mean(), r_raw.std()), "r-", lw=1.5, label="Normal fit")
ax[1].set_title("Histogram — daily log returns"); ax[1].legend()
sm.qqplot(r_raw, line="s", ax=ax[2], markersize=3); ax[2].set_title("QQ plot — log returns")
savefig("03_hist_qq"); plt.show()

In [ ]:
jb_stat, jb_p = stats.jarque_bera(r_raw)
desc = pd.DataFrame({
    "Price (USD)": close_raw.describe(),
    "Log return": r_raw.describe(),
}).round(5)
extra = pd.DataFrame({
    "Price (USD)": [close_raw.skew(), close_raw.kurt()],
    "Log return": [r_raw.skew(), r_raw.kurt()],
}, index=["skewness", "excess kurtosis"]).round(4)
display(pd.concat([desc, extra]))
print(f"Jarque-Bera on returns: stat = {jb_stat:,.1f}, p = {jb_p:.3g}  ->",
      "reject normality (fat tails)" if jb_p < 0.05 else "cannot reject normality")
print(f"Annualised volatility : {r_raw.std() * np.sqrt(252) * 100:.1f}%")

## 3. Dealing with Missing Values
Futures do not trade on every weekday (exchange holidays). We re-index to a full **business-day** calendar so the series has a regular frequency (required by ARIMA/SARIMA), then **forward-fill** — the last known price is the best available value on a non-trading day.

In [ ]:
price = close_raw.asfreq("B")
n_missing = int(price.isna().sum())
print(f"Missing business days before filling: {n_missing} ({n_missing / len(price):.2%})")
print("Missing values in raw Close column  :", int(close_raw.isna().sum()))
price = price.ffill().dropna()
assert price.isna().sum() == 0

log_p = np.log(price)          # log price  (stabilises variance, returns become % changes)
ret   = log_p.diff().dropna()  # daily log return
print(f"Clean series: {len(price):,} business days, {price.index.min().date()} → {price.index.max().date()}")

## 4. Stationarity — ADF and KPSS tests
* **ADF**: H0 = unit root (non-stationary). Small p → stationary.
* **KPSS**: H0 = stationary. Small p → non-stationary.
Using both gives a cross-check: *stationary* means ADF rejects **and** KPSS does not.

In [ ]:
def stationarity_table(series_dict):
    rows = []
    for name, s in series_dict.items():
        s = s.dropna()
        adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
        kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
        verdict = ("stationary" if (adf_p < .05 and kpss_p >= .05)
                   else "non-stationary" if (adf_p >= .05 and kpss_p < .05)
                   else "inconclusive")
        rows.append([name, adf_stat, adf_p, kpss_stat, kpss_p, verdict])
    return pd.DataFrame(rows, columns=["series", "ADF stat", "ADF p", "KPSS stat", "KPSS p", "verdict"]).round(4)

stationarity = stationarity_table({"Price": price, "Log price": log_p, "Log return (d=1)": ret})
stationarity.to_csv("outputs/stationarity.csv", index=False)
stationarity

## 5. Transformations
Price → **log** (stabilise variance) → **first difference** (remove stochastic trend) = daily **log return**.
This project evaluates d = 1. Check the ADF/KPSS results before interpreting that choice; stationarity of a second difference alone does not prove the first difference is sufficient.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ax[0].plot(price, lw=.9); ax[0].set_title("Price (non-stationary)")
ax[1].plot(log_p, lw=.9, color="darkgoldenrod"); ax[1].set_title("Log price (still has a trend)")
ax[2].plot(ret, lw=.6, color="teal"); ax[2].set_title("Log returns (stationary, but volatility clusters)")
ax[2].plot(ret.rolling(21).std() * 3, color="crimson", lw=1, label="±3 x 21-day rolling std")
ax[2].plot(-ret.rolling(21).std() * 3, color="crimson", lw=1); ax[2].legend(loc="upper left")
savefig("04_transformations"); plt.show()

d2 = ret.diff().dropna()
print("ADF p-value, 2nd difference:", round(adfuller(d2, autolag="AIC")[1], 6), "-> compare with the first-difference tests; do not choose d using this test alone")

## 6. Seasonality check & decomposition
Gold has no strong calendar seasonality (though jewellery demand creates mild festive-season effects). We *test* rather than assume: seasonal strength (Hyndman) for weekly / monthly / annual periods, day-of-week and month-of-year patterns.

In [ ]:
def seasonal_strength(s, period):
    dec = seasonal_decompose(s, model="additive", period=period)
    resid, seas = dec.resid.dropna(), dec.seasonal.loc[dec.resid.dropna().index]
    return max(0, 1 - resid.var() / (resid + seas).var())

strength = pd.DataFrame({
    "period (business days)": [5, 21, 252],
    "meaning": ["weekly", "monthly", "annual"],
    "seasonal strength (0-1)": [seasonal_strength(log_p, p) for p in (5, 21, 252)],
}).round(4)
strength

In [ ]:
dec = seasonal_decompose(log_p, model="additive", period=252)
fig = dec.plot(); fig.set_size_inches(12, 8); fig.suptitle("Additive decomposition of log price (annual period = 252)", y=1.01)
savefig("05_decomposition"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
dow = ret.groupby(ret.index.dayofweek).mean() * 100
ax[0].bar(["Mon", "Tue", "Wed", "Thu", "Fri"], dow.values, color="goldenrod")
ax[0].set_title("Mean daily return by weekday (%)")
mon = ret.groupby(ret.index.month).mean() * 100
ax[1].bar(mon.index, mon.values, color="steelblue"); ax[1].set_xticks(range(1, 13))
ax[1].set_title("Mean daily return by month (%)")
savefig("06_seasonal_patterns"); plt.show()

# is the month effect statistically distinguishable? (one-way ANOVA across months)
groups = [g.values for _, g in ret.groupby(ret.index.month)]
f, p = stats.f_oneway(*groups)
print(f"ANOVA of returns across months: F = {f:.2f}, p = {p:.3f} ->",
      "month effect detected" if p < .05 else "no significant month effect")

## 7. ACF and PACF

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(ret, lags=40, ax=ax[0, 0], title="ACF — log returns")
plot_pacf(ret, lags=40, ax=ax[0, 1], method="ywm", title="PACF — log returns")
plot_acf(ret ** 2, lags=40, ax=ax[1, 0], title="ACF — squared returns (volatility)")
plot_pacf(ret ** 2, lags=40, ax=ax[1, 1], method="ywm", title="PACF — squared returns")
savefig("07_acf_pacf"); plt.show()

## 8. Identify the model family (AR / MA / ARMA / ARIMA / SARIMA)
Rules of thumb used:

| ACF of differenced series | PACF | Suggests |
|---|---|---|
| decays slowly / geometrically | cuts off after lag p | AR(p) |
| cuts off after lag q | decays | MA(q) |
| both decay | both decay | ARMA(p, q) |
| spikes at seasonal lags (5, 10, …) | | add seasonal terms → SARIMA |

Below we count how many ACF/PACF lags breach the 95% band (≈ 5% of lags exceed it by chance alone) and run Ljung-Box.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
nl = 40
a = acf(ret, nlags=nl)[1:]; pa = pacf(ret, nlags=nl, method="ywm")[1:]
band = 1.96 / np.sqrt(len(ret))
sig_a  = [i + 1 for i, v in enumerate(a)  if abs(v) > band]
sig_pa = [i + 1 for i, v in enumerate(pa) if abs(v) > band]
print(f"95% band = ±{band:.4f}")
print(f"ACF  significant lags : {sig_a}   ({len(sig_a)}/{nl}; ~{.05 * nl:.0f} expected by chance)")
print(f"PACF significant lags : {sig_pa}   ({len(sig_pa)}/{nl})")
print("Seasonal lags (5,10,15,20) significant in ACF:", [l for l in (5, 10, 15, 20) if l in sig_a])

lb = acorr_ljungbox(ret, lags=[10, 20, 40], return_df=True)
lb2 = acorr_ljungbox(ret ** 2, lags=[10, 20, 40], return_df=True)
display(pd.concat({"Ljung-Box on returns": lb, "Ljung-Box on squared returns": lb2}, axis=1).round(4))

print("\nDecision: d = 1 (confirmed in §4/§5). If returns show little linear autocorrelation, expect a low-order ARIMA(p,1,q)")
print("(p, q <= 2 typically). Seasonal terms are only justified if seasonal lags are clearly significant — tested formally in §10 with a SARIMA comparison.")

## 9. Lag identification and AIC / BIC model selection
Grid-search ARIMA(p, 1, q) with a drift term on **log price**, p, q ∈ {0 … 4}. Models are estimated on the **training set only** (the last `TEST_DAYS` are held out) to avoid leakage.

* **AIC** = 2k − 2 ln L  → favours predictive accuracy
* **BIC** = k ln n − 2 ln L → penalises complexity harder, favours parsimony

In [ ]:
train_log, test_log = log_p.iloc[:-TEST_DAYS], log_p.iloc[-TEST_DAYS:]
print(f"Train: {len(train_log)} days ({train_log.index[0].date()} → {train_log.index[-1].date()})")
print(f"Test : {len(test_log)} days ({test_log.index[0].date()} → {test_log.index[-1].date()})")

rows = []
for p in range(MAX_P + 1):
    for q in range(MAX_Q + 1):
        try:
            m = ARIMA(train_log, order=(p, 1, q), trend="t").fit()
            rows.append([p, q, m.aic, m.bic, m.llf])
        except Exception as e:
            rows.append([p, q, np.nan, np.nan, np.nan])
grid = pd.DataFrame(rows, columns=["p", "q", "AIC", "BIC", "logLik"]).dropna()

best_aic = grid.loc[grid["AIC"].idxmin()]
best_bic = grid.loc[grid["BIC"].idxmin()]
print(f"\nBest by AIC: ARIMA({int(best_aic.p)},1,{int(best_aic.q)})  AIC = {best_aic.AIC:,.2f}")
print(f"Best by BIC: ARIMA({int(best_bic.p)},1,{int(best_bic.q)})  BIC = {best_bic.BIC:,.2f}")
display(grid.sort_values("BIC").head(8).round(2).reset_index(drop=True))
grid.to_csv("outputs/arima_grid_search.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for a_, col in zip(ax, ["AIC", "BIC"]):
    piv = grid.pivot(index="p", columns="q", values=col)
    im = a_.imshow(piv.values - np.nanmin(piv.values), cmap="YlOrBr_r")
    a_.set_xticks(range(len(piv.columns))); a_.set_xticklabels(piv.columns)
    a_.set_yticks(range(len(piv.index)));   a_.set_yticklabels(piv.index)
    a_.set_xlabel("q"); a_.set_ylabel("p"); a_.set_title(f"{col} − min({col})  (darker = better)")
    a_.grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            a_.text(j, i, f"{piv.values[i, j] - np.nanmin(piv.values):.0f}", ha="center", va="center", fontsize=8)
savefig("08_aic_bic_heatmap"); plt.show()

## 10. Out-of-sample evaluation
**One-step-ahead forecasts** over the test window: parameters are estimated on the training set, then the model is *updated* with each new observation (no re-estimation) — this mimics how it would run in production.
Baselines/competitors: **Naive** (tomorrow = today, i.e. a random walk), ARIMA (best AIC), ARIMA (best BIC), **SARIMA** (best BIC order + seasonal (1,0,1)₅).

In [ ]:
def one_step_forecast(order, seasonal_order=(0, 0, 0, 0)):
    fit = ARIMA(train_log, order=order, seasonal_order=seasonal_order, trend="t").fit()
    upd = fit.append(test_log, refit=False)
    pred = upd.get_prediction(start=len(train_log), end=len(log_p) - 1)
    return fit, pd.Series(np.exp(np.asarray(pred.predicted_mean)), index=test_log.index)

actual = price.loc[test_log.index]
prev   = price.shift(1).loc[test_log.index]

order_aic = (int(best_aic.p), 1, int(best_aic.q))
order_bic = (int(best_bic.p), 1, int(best_bic.q))

preds = {"Naive (random walk)": prev}
fit_aic, preds[f"ARIMA{order_aic} [AIC]"] = one_step_forecast(order_aic)
fit_bic, preds[f"ARIMA{order_bic} [BIC]"] = one_step_forecast(order_bic)
fit_sar, preds[f"SARIMA{order_bic}x(1,0,1,{SEAS_PERIOD})"] = one_step_forecast(order_bic, (1, 0, 1, SEAS_PERIOD))

def metrics(pred):
    e = actual - pred
    d_true = np.sign(actual - prev); d_pred = np.sign(pred - prev)
    mask = d_pred != 0
    return {"RMSE": np.sqrt((e ** 2).mean()), "MAE": e.abs().mean(),
            "MAPE %": (e.abs() / actual).mean() * 100,
            "Directional acc %": (d_true[mask] == d_pred[mask]).mean() * 100 if mask.any() else np.nan}

def diebold_mariano(e_model, e_base):
    d = e_model ** 2 - e_base ** 2           # negative mean -> model better than baseline
    # HAC standard error allows serial correlation in loss differences.
    if np.allclose(d, 0):
        return 0.0, 1.0
    test = sm.OLS(np.asarray(d), np.ones((len(d), 1))).fit(
        cov_type="HAC", cov_kwds={"maxlags": max(1, int(len(d) ** (1/3)))})
    return float(test.tvalues[0]), float(test.pvalues[0])

results = pd.DataFrame({k: metrics(v) for k, v in preds.items()}).T
base_err = actual - preds["Naive (random walk)"]
results["DM p-value vs naive"] = [np.nan if k.startswith("Naive") else diebold_mariano(actual - v, base_err)[1]
                                  for k, v in preds.items()]
results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(price.iloc[-TEST_DAYS - 40:], color="black", lw=1.4, label="Actual")
for (k, v), c in zip(list(preds.items())[1:], ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(v, lw=1, ls="--", color=c, label=k)
ax.axvline(test_log.index[0], color="grey", ls=":"); ax.set_title("One-step-ahead forecasts on the hold-out window")
ax.legend(fontsize=8); savefig("09_test_forecasts"); plt.show()
results.to_csv("outputs/model_comparison.csv")

In [ ]:
# Multi-step: forecast the whole test window from the training end (no updating)
H = TEST_DAYS
fc = fit_bic.get_forecast(H)
mu, ci = np.exp(fc.predicted_mean.values), np.exp(fc.conf_int(alpha=.05).values)
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(price.iloc[-TEST_DAYS - 120:], color="black", lw=1.2, label="Actual")
ax.plot(test_log.index, mu, color="tab:green", label=f"ARIMA{order_bic} {H}-day forecast")
ax.fill_between(test_log.index, ci[:, 0], ci[:, 1], color="tab:green", alpha=.2, label="95% interval")
ax.legend(); ax.set_title(f"{H}-day-ahead forecast made at the end of training")
savefig("10_multistep_forecast"); plt.show()
cover = ((actual.values >= ci[:, 0]) & (actual.values <= ci[:, 1])).mean()
print(f"Empirical coverage of the 95% interval on the test window: {cover:.1%}")

## 11. Residual diagnostics (best-BIC ARIMA)

In [ ]:
resid = fit_bic.resid.iloc[1:]       # drop the first (diffuse-initialisation) residual
fig = fit_bic.plot_diagnostics(figsize=(12, 8), lags=30)
savefig("11_residual_diagnostics"); plt.show()

lbr = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
lbr2 = acorr_ljungbox(resid ** 2, lags=[10, 20], return_df=True)
arch_stat, arch_p, _, _ = het_arch(resid, nlags=10)
jb, jbp = stats.jarque_bera(resid)
diag = pd.DataFrame({
    "test": ["Ljung-Box (resid), lag 10", "Ljung-Box (resid), lag 20",
             "Ljung-Box (resid²), lag 10", "ARCH-LM, 10 lags", "Jarque-Bera (normality)"],
    "p-value": [lbr["lb_pvalue"].iloc[0], lbr["lb_pvalue"].iloc[1], lbr2["lb_pvalue"].iloc[0], arch_p, jbp],
})
diag["reading"] = np.where(diag["p-value"] < .05, "FAIL (assumption violated)", "pass")
diag.to_csv("outputs/residual_diagnostics.csv", index=False)
diag.round(4)

### 11b. GARCH(1,1) — modelling the volatility clustering
Financial returns usually pass the *mean* checks (no autocorrelation) but **fail the ARCH test**: variance changes over time, so ARIMA's innovation variance is constant, which can miscalibrate its widening forecast intervals. A **GARCH(1,1)** with Student-t errors models the time-varying variance, and we use it to build a volatility-aware interval.

In [ ]:
r100 = ret.loc[train_log.index[1:]] * 100
garch = arch_model(r100, mean="Constant", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
print(garch.summary().tables[1]); print(garch.summary().tables[2])
alpha, beta = garch.params["alpha[1]"], garch.params["beta[1]"]
print(f"\nPersistence alpha + beta = {alpha + beta:.4f}  (close to 1 => volatility shocks fade slowly)")

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(garch.conditional_volatility, color="crimson", lw=.9)
ax.set_title("GARCH(1,1) conditional daily volatility (%)")
savefig("12_garch_volatility"); plt.show()

## 12. Final model, forecast and deployment
Re-fit the chosen ARIMA on **all** data, forecast `HORIZON` business days, and show two intervals: the standard ARIMA one and a GARCH-based one (wider when current volatility is high).

In [ ]:
FINAL_ORDER = order_bic
final = ARIMA(log_p, order=FINAL_ORDER, trend="t").fit()
print(final.summary().tables[1])

fc = final.get_forecast(HORIZON)
f_idx = pd.bdate_range(price.index[-1] + pd.offsets.BDay(1), periods=HORIZON)
f_mean = pd.Series(np.exp(fc.predicted_mean.values), index=f_idx)
f_ci = pd.DataFrame(np.exp(fc.conf_int(alpha=.05).values), index=f_idx, columns=["arima_lo", "arima_hi"])

# Approximate GARCH interval (normal quantile, not an exact Student-t interval): variance of the h-day cumulative return = sum of daily conditional variances
g_all = arch_model(ret * 100, mean="Constant", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
gv = g_all.forecast(horizon=HORIZON).variance.iloc[-1].values       # % squared
cum_sd = np.sqrt(np.cumsum(gv)) / 100
log_mean = np.log(f_mean.values)
f_ci["garch_lo"] = np.exp(log_mean - 1.96 * cum_sd)
f_ci["garch_hi"] = np.exp(log_mean + 1.96 * cum_sd)

forecast_df = pd.concat([f_mean.rename("forecast"), f_ci], axis=1).round(2)
forecast_df.index.name = "date"
forecast_df.to_csv("outputs/forecast_next_30_days.csv")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(price.iloc[-250:], color="black", lw=1.3, label="History")
ax.plot(f_mean, color="tab:green", lw=2, label=f"ARIMA{FINAL_ORDER} forecast")
ax.fill_between(f_idx, f_ci["arima_lo"], f_ci["arima_hi"], color="tab:green", alpha=.18, label="ARIMA 95% interval")
ax.plot(f_idx, f_ci["garch_lo"], color="crimson", ls="--", lw=1); ax.plot(f_idx, f_ci["garch_hi"], color="crimson", ls="--", lw=1, label="Approximate GARCH normal 95% band")
ax.set_title(f"{HORIZON}-business-day gold price forecast"); ax.legend(); ax.set_ylabel("USD / oz")
savefig("13_final_forecast"); plt.show()
forecast_df.head(10)

In [ ]:
# ---- save model + metadata for the Streamlit app ----
final.save("models/gold_arima.pkl")
best_row = results.loc[[k for k in results.index if "[BIC]" in k][0]]
naive_row = results.loc["Naive (random walk)"]
meta = {
    "ticker": TICKER, "data_source": DATA_SOURCE,
    "order": list(FINAL_ORDER), "trend": "t",
    "training_ticker": TICKER,
    "trained_until": str(price.index[-1].date()), "n_obs": int(len(log_p)),
    "test_days": TEST_DAYS, "test_end": str(test_log.index[-1].date()),
    "test_rmse_model": float(best_row["RMSE"]), "test_rmse_naive": float(naive_row["RMSE"]),
    "test_mape_model": float(best_row["MAPE %"]), "test_mape_naive": float(naive_row["MAPE %"]),
}
Path("models/model_meta.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

In [ ]:
# ---- reusable forecasting function (this is the 'deployed' model) ----
from statsmodels.tsa.arima.model import ARIMAResults

def forecast_gold(n_days=5, new_prices=None, model_path="models/gold_arima.pkl"):
    # Load the saved model, optionally absorb newer closing prices (no refit), forecast n business days.
    model = ARIMAResults.load(model_path)
    if new_prices is not None:
        model = model.append(np.log(pd.Series(new_prices).asfreq("B").ffill()), refit=False)
    fc = model.get_forecast(n_days)
    out = pd.DataFrame(np.exp(fc.conf_int().values), index=fc.predicted_mean.index, columns=["lower_95", "upper_95"])
    out.insert(0, "forecast", np.exp(fc.predicted_mean.values))
    out.index.name = "date"
    return out.round(2)

forecast_gold(5)

**Run the web app:** `streamlit run app.py` (reads `models/model_meta.json` and refits on live data).

## 13. Summary of results (auto-generated)

In [ ]:
winner = results["RMSE"].idxmin()
print("=" * 72)
print(f"Data source           : {DATA_SOURCE}")
print(f"Sample                : {price.index[0].date()} → {price.index[-1].date()} ({len(price):,} business days)")
p_price, p_ret = adfuller(log_p, autolag="AIC")[1], adfuller(ret, autolag="AIC")[1]
print(f"Stationarity (ADF)    : log price p = {p_price:.3f} ({'non-stationary' if p_price >= .05 else 'stationary'}); "
      f"log returns p = {p_ret:.3g} ({'stationary' if p_ret < .05 else 'non-stationary'}) => d = 1")
print(f"Chosen model          : ARIMA{FINAL_ORDER} + drift (min BIC); min-AIC was ARIMA{order_aic}")
print(f"Seasonality           : weekly/monthly/annual strength = {strength['seasonal strength (0-1)'].round(3).tolist()}")
print(f"Best hold-out RMSE    : {winner} ({results.loc[winner, 'RMSE']:.2f} USD)")
print(f"Naive RMSE            : {results.loc['Naive (random walk)', 'RMSE']:.2f} USD")
print(f"ARCH effects (resid)  : ARCH-LM p = {arch_p:.4g} -> {'present: use GARCH for intervals' if arch_p < .05 else 'not detected'}")
print("=" * 72)

### Interpretation guide (edit once you see your own numbers)
1. **Is gold a random walk?** Compare ARIMA with the naive baseline in §10. If the Diebold-Mariano p-value is > 0.05, ARIMA is *not significantly better* than "tomorrow = today". That is a legitimate finding — it is consistent with a random-walk benchmark, but does not establish market efficiency — not a failure.
2. **Why still use ARIMA?** The drift term and the forecast **interval** are what add value over the naive point forecast; the interval quantifies uncertainty.
3. **Volatility clustering:** if the ARCH-LM test rejects (§11), the ARIMA interval is too narrow in turbulent periods and too wide in calm ones — hence the GARCH-based interval.
4. **Seasonality:** if seasonal strength is near 0 and SARIMA does not improve the hold-out RMSE/BIC, the extra seasonal parameters are unjustified — prefer the simpler ARIMA.
5. **Limitations:** a univariate model ignores drivers of gold (USD index, real interest rates, inflation, geopolitics). Natural extension: **VAR** (Unit 8) with the dollar index and 10-year yield. Not financial advice.

### References (cite these in your report)
* Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed.), OTexts.
* Brockwell & Davis, *Introduction to Time Series and Forecasting*.
* Seabold & Perktold (2010), *statsmodels*, SciPy Proceedings. · Sheppard, *arch* package.
* Engle (1982) ARCH; Bollerslev (1986) GARCH. · Diebold & Mariano (1995).